In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


df = pd.read_csv('./data11.csv')
X = df.drop(columns="grade_class")
y = df["grade_class"].eq("high").astype(int).to_numpy()

print(f"Наблюдений: {len(df)}")
print(df["grade_class"].value_counts().to_string())

Наблюдений: 3489
grade_class
low     2659
high     830


In [2]:
numeric = X.select_dtypes(include="number").columns
categorical = X.select_dtypes(exclude="number").columns

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
])

model = make_pipeline(preprocess, LogisticRegression(C=np.inf, max_iter=1000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []
for train_idx, val_idx in cv.split(X, y):
    model.fit(X.iloc[train_idx], y[train_idx])
    probability = model.predict_proba(X.iloc[val_idx])[:, 1]
    folds.append((y[val_idx], probability))

In [3]:
thresholds = np.linspace(0, 1, 10_001)

recalls = []
for y_val, probability in folds:
    positive_probabilities = np.sort(probability[y_val == 1])
    below_threshold = np.searchsorted(positive_probabilities, thresholds)
    recalls.append(1 - below_threshold / len(positive_probabilities))

mean_recall = np.mean(recalls, axis=0)
error = np.abs(mean_recall - 0.8)
best_indices = np.flatnonzero(np.isclose(error, error.min(), rtol=0, atol=1e-12))
threshold = thresholds[best_indices[-1]]

print(f"Порог C = {threshold:.4f}")
print(f"Средний recall при пороге 0.5 = {mean_recall[5000]:.6f}")

Порог C = 0.1743
Средний recall при пороге 0.5 = 0.183133


In [4]:
rows = []
for y_val, probability in folds:
    prediction = (probability >= threshold).astype(int)
    rows.append({
        "accuracy": accuracy_score(y_val, prediction),
        "precision": precision_score(y_val, prediction, zero_division=0),
        "recall": recall_score(y_val, prediction),
    })

results = pd.DataFrame(rows, index=pd.Index(range(1, 6), name="Фолд"))
results.loc["Среднее"] = results.mean()
print(results.to_string(float_format=lambda value: f"{value:.6f}"))

         accuracy  precision   recall
Фолд                                 
1        0.567335   0.341121 0.879518
2        0.561605   0.330097 0.819277
3        0.561605   0.312834 0.704819
4        0.522923   0.306265 0.795181
5        0.540890   0.316667 0.801205
Среднее  0.550871   0.321397 0.800000
